# Algorytmika i matematyka uczenia maszynowego 
## Laboratorium 11

### Zadanie 1

Zaimplementuj systemu rekomendacji filmów w oparciu o indeks Jaccarda. System ma za zadanie zwrócić listę filmów sugerowany dla podanego użytownika.

Dane zostały pobrane z serwisu Kaggle z https://www.kaggle.com/datasets/gargmanas/movierecommenderdataset

Zbiór zawiera dwa pliki:
- `movies.csv` lista filmów wraz z ich identyfikatorami
- `ratings.csv` lista ocen filmów przez użytkowników

**Zadanie:**
* Wczytaj oba pliki.
* Zamień wszystkie oceny użytkownika na wartość 1 (zastosuj próg okreśjący czy film się podobał czy nie np. 3).
* Stwórz macierz ocen użytkowników w której wierszach będą użytkownicy, a w kolumnach filmy. Wartość w macierzy jest flagą mówiącą czy użytkownikowi film się podobał czy nie. 
* Wypełnij brakujące wartości zerami.
* Utwórz macierz podobieństwa Jaccarda pomiędzy użytkownikami (każdy z każdym).
    - Możesz wykorzystać funkcję [jaccard](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.jaccard.html) z biblioteki scipy.
* Zaimplementuj funkcję która dla podanego użytkownika zwróci listę sugerowanych filmów.
    - Funkcja powinna zwrócić listę filmów które nie były ocenione przez użytkownika, a które są rekomendowane dla niego.



In [24]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform


movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

ratings['rating'] = (ratings['rating'] > 2.5).astype(int)

movies_id = movies['movieId'].unique()
users_id = ratings['userId'].unique()
user_ratings = np.zeros((len(users_id), len(movies_id)))

movie_to_index = {movie_id: idx for idx, movie_id in enumerate(movies_id)}
user_to_index = {user_id: idx for idx, user_id in enumerate(users_id)}

for _, row in ratings.iterrows():
    if row['rating'] == 1:
        user_idx = user_to_index[row['userId']]
        movie_idx = movie_to_index[row['movieId']]
        user_ratings[user_idx][movie_idx] = 1

user_ratings_df = pd.DataFrame(user_ratings, index=users_id, columns=movies_id)
print(user_ratings_df)

jaccard_distances = pdist(user_ratings, metric='jaccard')

jaccard_similarity = 1 - squareform(jaccard_distances)
print(jaccard_similarity)

def recommend_movies_for_user(user_id, user_ratings, user_to_index, movies_id, jaccard_similarity, top_k=5):
    user_idx = user_to_index[user_id]
    
    # Podobieństwa do innych użytkowników
    similarities = jaccard_similarity[user_idx]
    
    # Indeksy top K najbardziej podobnych użytkowników (pomijając siebie)
    similar_users_idx = np.argsort(-similarities)
    similar_users_idx = [idx for idx in similar_users_idx if idx != user_idx][:top_k]
    
    # Filmy ocenione pozytywnie przez podobnych użytkowników
    similar_users_ratings = user_ratings[similar_users_idx]
    scores = np.sum(similar_users_ratings, axis=0)
    
    # Filmy jeszcze nie ocenione przez użytkownika
    unseen_by_user = user_ratings[user_idx] == 0
    
    # Rekomendowane filmy: posortowane wg liczby rekomendacji
    recommended_indices = np.argsort(-scores * unseen_by_user)
    recommended_movie_ids = [movies_id[idx] for idx in recommended_indices if unseen_by_user[idx] and scores[idx] > 0]
    
    return recommended_movie_ids

user_id = 1  # dowolny userId z danych
recommended = recommend_movies_for_user(user_id, user_ratings, user_to_index, movies_id, jaccard_similarity)
print("Polecane filmy dla użytkownika:", recommended)

recommended_titles = movies[movies['movieId'].isin(recommended)]['title'].tolist()
print("Tytuły polecanych filmów:")
for title in recommended_titles:
    print(title)


     1       2       3       4       5       6       7       8       9       \
1       1.0     0.0     1.0     0.0     0.0     1.0     0.0     0.0     0.0   
2       0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3       0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4       0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5       1.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
..      ...     ...     ...     ...     ...     ...     ...     ...     ...   
606     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
607     1.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
608     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
609     1.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
610     1.0     0.0     0.0     0.0     0.0     1.0     0.0     0.0     0.0   

     10      ...  193565  193567  193571  193573  1



### Zadanie 2


Algorytm MinHash na przykładzie wykrywania plagiatów

Wykonaj kolejno następujące kroki:

1. Pobierz 8 akapitów tekstu (nie za krótkich), każdy o różnej tematyce (mogą być np. z różnych haseł Wikipedii), trzymaj się jednego języka (np. PL lub ENG). Wklej je do jednego pliku tekstowego, z linią wolną jako separatorem.

2. Skopiuj wybrane 2-3 akapity i ręcznie nieco zmodyfikuj.

> Przykład (z hasła https://pl.wikipedia.org/wiki/Fryderyk_Chopin):

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych polskich kompozytorów w historii. Był jednym z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu. Elementem charakterystycznym dla utworów Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców stylistycznych polskiej muzyki ludowej.```

↓↓↓ Zmieniono na ↓↓↓

```Jest uważany za jednego z najwybitniejszych kompozytorów romantycznych, a także za jednego z najważniejszych kompozytorów polskich w historii.  Był jednym  z najsłynniejszych pianistów swoich czasów, często nazywany poetą fortepianu! Elementem charakterystycznym dla utworów Fryderyka Chopina jest pogłębiona ekspresja oraz czerpanie z wzorców polskiej muzyki ludowej.```

Otrzymasz zatem w pliku tekstowym 10 lub 11 akapitów tekstu (kolejność dowolna, te „splagiatowane” nie muszą być na końcu).

3. Z poziomu skryptu: wczytaj wszystkie akapity z pliku. Zbuduj 100 "losowych" funkcji haszujących.

> Sugestia: funkcją "bazową" jest po prostu `hash(...)`. Zakładamy 64-bitową wersję Pythona 3.x, wtedy `hash(...)` jest 64-bitowy.

Na liście seeds umieszczamy 100 losowych liczb 64-bitowych. Aby obliczyć $i$-ty hash dla ciągu 
należy wykonać `hash(s) ^ seeds[i]` (użycie operatora XOR).

4. Przyjmij niewielką wartość $Q$ (np. 15) i dla każdego akapitu
    - oznacz jego długość przez $n$,
    - dla każdej ze 100 funkcji haszujących policz hasza w przesuwnym oknie tekstu o długości $Q$ znaków (czyli łącznie mamy $n - Q + 1$ wartości hasza); zapamiętaj MINIMUM z tych $n - Q + 1$
 wartości.
Na wyjściu mamy zatem (dla 11 akapitów) 11 * 100 wartości haszy.

5. Rozważ pary akapitów "każdy z każdym". Jeśli dla danej pary co najmniej (np.) 30 haszy jest wspólnych, to uważamy akapity za podobne (być może plagiat) i wyświetlamy na ekranie.

6. Wyświetl czas obliczeń (powinien wynosić mniej niż 0.5s).

7. Poeksperymentuj z liczbą użytych funkcji haszujących, wartością, stopniem modyfikacji oryginalnych akapitów tekstu, progiem detekcji akapitów podobnych.

In [ ]:
# UZUPEŁNIJ